# 107 — Evaluación de fidelidad, cobertura y atribución

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Cuatro elementos por interacción: pregunta `q`, contexto `C`, respuesta `a`, referencia
`g`. Cada métrica RAGAS (arXiv:2309.15217) compara un par distinto:

- **Faithfulness** (`a` vs `C`, sin ground truth): descomponer `a` en afirmaciones
  atómicas y contar cuántas implica el contexto: `implicadas / total`. Monitoreable en
  producción.
- **Answer relevancy** (`a` vs `q`): ¿responde lo preguntado?
- **Context recall** (`C` vs `g`): ¿el contexto cubre lo necesario? Si es bajo, falló el
  retriever y ningún generador lo compensa.
- **Context precision** (`C` vs `q`): ¿lo recuperado es relevante y está bien ordenado?

**Atribución** (AIS arXiv:2112.12870, ALCE arXiv:2305.14627): *citation recall* (¿cada
afirmación tiene cita que la respalda?) y *citation precision* (¿cada cita emitida
implica su afirmación?). Se reportan juntas: forzar citas sube una y hunde la otra.

**LLM-as-judge**: escala la evaluación pero tiene sesgos (posición, verbosidad,
autopreferencia); se calibra contra una muestra anotada por humanos antes de confiar
en sus números.

## 🧮 Ejemplo de referencia

```text
C: [1] "El museo se inauguró en octubre de 1997."
   [2] "El edificio fue diseñado por Frank Gehry."
a: "Diseñado por Gehry [2], se inauguró en 1997 [1] y recibió un millón
    de visitantes en su primer año."

s1 Gehry        → implicada, cita ✓      s2 1997 → implicada, cita ✓
s3 visitantes   → NO implicada, sin cita

faithfulness = 2/3          citation recall = 2/3
citation precision = 2/2    context recall = context precision = 1.0
```

Diagnóstico: retriever perfecto; el fallo es solo del generador (s3 salió de la memoria
paramétrica). Las métricas por componente dicen *qué* arreglar.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("evaluation", seed=107)
show(result)


## Reflexión

1. Construye (mentalmente) un caso con faithfulness = 1.0 y respuesta factualmente falsa. ¿Qué componente del pipeline falló y qué métrica lo habría detectado?
2. ¿Por qué faithfulness puede monitorearse sobre tráfico real de producción pero context recall no? ¿Qué se necesita para cada una?
3. Si tu juez LLM concuerda con humanos en el 78 % de los veredictos de entailment, ¿qué significa un cambio de faithfulness de 0.85 a 0.88 entre dos versiones del sistema?